In [1]:
import pandas as pd
import sqlite3

In [6]:
df = pd.read_csv("../googleplaystore.csv")
conn = sqlite3.connect("games.db")
df.to_sql("games", conn, if_exists="replace", index=False)

10841

In [8]:
result = pd.read_sql_query("""
SELECT * FROM games
""", conn)
result

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10836,Sya9a Maroc - FR,FAMILY,4.5,38,53M,"5,000+",Free,0,Everyone,Education,"July 25, 2017",1.48,4.1 and up
10837,Fr. Mike Schmitz Audio Teachings,FAMILY,5.0,4,3.6M,100+,Free,0,Everyone,Education,"July 6, 2018",1.0,4.1 and up
10838,Parkinson Exercices FR,MEDICAL,NaN,3,9.5M,"1,000+",Free,0,Everyone,Medical,"January 20, 2017",1.0,2.2 and up
10839,The SCP Foundation DB fr nn5n,BOOKS_AND_REFERENCE,4.5,114,Varies with device,"1,000+",Free,0,Mature 17+,Books & Reference,"January 19, 2015",Varies with device,Varies with device


To identify which categories have the most installs among paid apps.

In [27]:
result = pd.read_sql_query("""
SELECT App, Category, Installs FROM games
WHERE Type = "Paid"
GROUP BY Category
ORDER BY SUM(Installs) DESC
""", conn)
result

,App,Category,Installs
0,Toca Mystery House,FAMILY,5000
1,The Game of Life,GAME,100000
2,Beautiful Widgets Pro,PERSONALIZATION,1000000
3,Facetune - For Free,PHOTOGRAPHY,1000000
4,ADS-B Driver,TOOLS,100
5,aCalendar+ Calendar & Tasks,PRODUCTIVITY,100000
6,Puffin Browser Pro,COMMUNICATION,100000
7,Golfshot Plus: Golf GPS,SPORTS,50000
8,A41 WatchFace for Android Wear Smart Watch,LIFESTYLE,5000
9,Monash Uni Low FODMAP Diet,MEDICAL,100000


In [18]:
df[~df['Installs'].str.isnumeric()]['Category'].unique()

array(['10,000+', '500,000+', '5,000,000+', '50,000,000+', '100,000+',
       '50,000+', '1,000,000+', '10,000,000+', '5,000+', '100,000,000+',
       '1,000,000,000+', '1,000+', '500,000,000+', '50+', '100+', '500+',
       '10+', '1+', '5+', '0+', 'Free'], dtype=object)

The Installs column contains non-numeric values such as "10,000+" and "Free". These must be cleaned before any numerical operation can be performed. To fix this, str.replace() was used to remove "+" and "," characters, and replace "Free" with "0". The column was then converted to integer using astype(int).

In [22]:
df['Installs'] = df['Installs'].str.replace('+', '')
df['Installs'] = df['Installs'].str.replace(',', '')
df['Installs'] = df['Installs'].str.replace('Free', '0')
df['Installs'] = df['Installs'].astype(int)
df.to_sql("games", conn, if_exists="replace", index=False)

10841

In [25]:
result = pd.read_sql_query("""
SELECT Category, SUM(Installs) FROM games
WHERE Type = "Paid"
GROUP BY Category
ORDER BY SUM(Installs) DESC
""", conn)
result

,Category,SUM(Installs)
0,FAMILY,31271814
1,GAME,21099965
2,PERSONALIZATION,5258794
3,PHOTOGRAPHY,3978740
4,TOOLS,1727441
5,PRODUCTIVITY,1412055
6,COMMUNICATION,1360050
7,SPORTS,1243815
8,LIFESTYLE,1179110
9,MEDICAL,1020033
